In [ ]:
import numpy as np
import scipy
import arviz as az
import xarray as xr
import pymc as pm
import matplotlib.pyplot as plt
plt.rcParams["figure.autolayout"] = True 

# Markov Chain Monte Carlo (MCMC)

* Readings: 
    * McElreath: Chapter 9 
    * (Extra) John K. Kruschke. Doing Bayesian Data Analysis, 2nd Edition. 2015: Chapters 9 and 14.
        * Accessible online via KU library: https://soeg.kb.dk/permalink/45KBDK_KGL/1pioq0f/alma99123031512905763

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Agenda

* Metropolis Algorithm
* Gibbs Algorithm
* Hamiltonian Monte Carlo
* MCMC Diagnostics

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Bayesian Inference (Recall)

* We are interested in computing the posterior distribution of some parameter(s) $\theta$, given some data $D$

* We use Bayes rule

$$
P(\theta \mid D) = \frac{P(D \mid \theta)P(\theta)}{\int_\theta P(D \mid \theta)P(\theta) d\theta}
$$

* However, this is intractable in practice for most interesting models

* Instead, we have been using Markov Chain Monte Carlo (MCMC) methods to compute an approximation of $P(\theta \mid D)$

* Today (1): How these methods work?

* Today (2): How to diagnose the quality of the approximation produced by MCMC methods?

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# MCMC Methods

<img style="float: right; height: 300px; padding-right: 5mm; padding-left: 15mm;" src="silly-walk.gif">

* Instead of computing the posterior directly, MCMC algorithm samples from it

* Generated samples approximate the posterior
    
    * They can be used to compute approximations for mode, HDI, mean, density plots, etc.

* Samples are generated by performing a *random walk*

* The groundbreaking part is that MCMC methods can sample from the posterior without an analytical solution for the normalizing factor $\int_\theta P(D \mid \theta)P(\theta) d\theta ~$   🤯
    
    * **[Question]: Can we evaluate $P(\theta)$ and $P(D \mid \theta)$ (for a given D and $\theta$)?**

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Metropolis Algorithm

<img style="float: right; padding-left: 2cm;" src="nicholas_metropolis.png">

* Perform a **random walk in the parameter space**, favoring states that have higher posterior probability density
* At each iteration a **proposal distribution** decides what next position we should consider
* Then we accept the proposal when it improves over our current density, or thanks to some small random luck
* The **proposal distribution is fixed** throughout the run of an algorithm, in principle, independent of the prior and likelihood.
* The **proposal distribution is centered on the current position** (usually a multivariate Gaussian distribution in the continuous case)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Good King Markov (a modified example, McElreath, Sect. 9.1)

* King Markov needs to visit the set of islands in his archipelago Kingdom proportional to their population

* Unlike in the textbook, we consider a non-circular archipelago
    * If the king is in island 1 it must move to island 2 or stay in 1, as there is no island 0
    * Similarly, if the king is in island 10, then it must move to island 9 or stay in island 10, as there are no islands 11, 12, ...
 
* Below we provide an instance of the Metropolis algorithm that accomplishes the King's goal

* Note the main elements of the Metropolis algorithm
    * Initial state
    * Proposal distribution
    * Density function proportional to target posterior distribution
    * Acceptance ratio

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

In [ ]:
np.random.seed(20240314)
num_samples = 3000
num_islands = 4
q_init  = 1 # could be any value between (1, num_islands)
samples = [q_init] # add init value to the beginning of the trace

# target distribution
# Density function proportional to island ID and 0 if islands outside the range
P = lambda x: x if x in range(1,num_islands+1) else 0

for i in range(num_samples):
    # current island
    q = samples[i] 

    # Randomly propose a new state (proposal)
    proposal = 1 if np.random.uniform(0,1) > .5 else -1
    q_prop = q + proposal

    # Move to proposed state? 
    q_next = q_prop if np.random.uniform(0,1) <= P(q_prop)/P(q) else q
    samples.append(q_next)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

Here we construct an inference data object that can be used to easily generate standard plots

In [ ]:
# construct an inference data object
dataset = xr.Dataset(
    {
        "θ": (["chain", "draw"], np.array([samples]))
    },
    coords = {
        "chain": (["chain"], np.arange(1)),
        "draw": (["draw"], np.arange(num_samples+1))
    }
)
trace = az.InferenceData(posterior=dataset)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

We plot the random walk

In [ ]:
(fig, ax) = plt.subplots(1)
ax.plot(trace.posterior.θ.sel(chain=0),np.arange(num_samples+1), '.-');
ax.set_xlabel('Island ID');
ax.xaxis.set_ticks(np.arange(1,num_islands+1));
ax.set_ylabel('Time step');

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

* The plot above is the same as the plot above.
* The left-hand-side of the posterior plot is a histogram with the frequency with which of each island was visited in the chain

In [ ]:
az.plot_trace(trace);

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Why does Metropolis work? (intuition)

* Let $P(\theta_i \to \theta_j)$ denote the probability of transitioning from $\theta_i$ to $\theta_j$. Then transition probabilities in our example are:

    * Transitioning to adjacent island
        
        * $P(\theta_i \to \theta_{j}) = 0.5 \cdot \min(\frac{P(\theta_{j})}{P(\theta_i)}, 1)$ where $j = i+1$ or $j=i-1$
    
    * Staying in current island

        * $P(\theta_i \to \theta_i) = 0.5 \cdot (1 - \min(\frac{P(\theta_{i-1})}{P(\theta_i)}, 1)) + 0.5 \cdot (1 - \min(\frac{P(\theta_{i+1})}{P(\theta_i)}, 1))$




* These transition probabilities define a Markov Chain
    
    * **Theorem**: The steady residence distribution produced by the Metropolis algorithm exists and equals $P$.
      
    * This theorem ensures that, with a long enough sample, the algorithm converges to an empirical representation of the target distribution $P$.

* For details see: Kruschke. _Doing Bayesian Data Analysis, Second Edition: A Tutorial with R, JAGS, and Stan._  Ch. 7, 

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

* Consider a simplified version of the problem with only 3 islands, denoted as $\theta_1, \theta_2, \theta_3$. Below we show the resulting Markov Chain.
    * Note that: $~~~\max(1 - \frac{P(\theta_{j})}{P(\theta_i)}, 0) = 1 - \min(\frac{P(\theta_{j})}{P(\theta_i)}, 1)$
 

<img style="height: 450px" src="mc-policitician-example.png">

* Recall that, for $P$ distribution in our example, we have that  $P(\theta_1) \leq P(\theta_2) \leq P(\theta_3)$.

* We can compute transition probabilities for every state. Below are some examples:


\begin{align*}
P(\theta_1 \to \theta_2) & = 0.5 \min \left(\frac{P(\theta_1)}{P(\theta_2)},1\right) \\
& = \text{if } P(\theta_1)\geq P(\theta_2) \text{ then } 0.5 \text{ else } 0.5  \frac{P(\theta_1)}{P(\theta_2)}\\
& = 0.5  \frac{P(\theta_1)}{P(\theta_2)}\\[4mm]
P(\theta_3 \to \theta_2) & = 0.5 \min \left(\frac{P(\theta_3)}{P(\theta_2)},1\right) \\
& = \text{if } P(\theta_3)\geq P(\theta_2) \text{ then } 0.5 \text{ else } 0.5  \frac{P(\theta_3)}{P(\theta_2)}\\
& = 0.5 \\[4mm]
P(\theta_2 \to \theta_2) & = 0.5 \max \left(1-\frac{P(\theta_1)}{P(\theta_2)},0\right)
+ 0.5 \max \left(1-\frac{P(\theta_3)}{P(\theta_2)},0\right)\\
& = 0.5 \left(1-\frac{P(\theta_1)}{P(\theta_2)}\right) + 0.5 \cdot 0 \\[2mm]
\end{align*}


* Note that all probabilities are constants larger than 0
    * Doing the exercise above for all cases shows the same results
    * This proves that the Markov Chain is *irreducible* and *aperiodic*
        * Irreducible: A Markov chain in which every state can be reached from every other state
        * Aperiodic: A Markov chain where no state is periodic. A state is periodic state if it is visited at regular intervals of time (every k-steps)
    * **If the Markov chain is irreducible and aperiodic, then there is a unique stationary distribution $P$**


<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Metropolis for Bayesian Inference of Posterior

* The target distribution function $P$ only needs to be *proportional* to the desired distribution

* We can use the $\mathit{likelihood} \times \mathit{prior}$ term in based theorem, as $Z$ is just a normalizing constant!
    
    * Recall that computing $Z$ was the main problem we had 🥳

* This is because the algorithm only works with the probability ratio between proposals
$$
\text{acceptance ratio} = 
\frac{P(\theta_2 \mid D)}{P(\theta_1 \mid D)} = \frac{P(D \mid \theta_2)p(\theta_2)/Z}{P(D \mid \theta_1)p(\theta_1)/Z} = \frac{P(D \mid \theta_1)p(\theta_2)}{P(D \mid \theta_1)p(\theta_1)}
$$

* Note that the structure of the algorithm is exactly the same as for the King's example without observations

* We only need to adapt the function proportional the target distribution $P$

    * (As we will see, the selection of proposal distribution is also important for speed of convergence)

* Below we show modification of the code where I use the population density function as likelihood, and I add a prior giving higher probability of visiting for islands 1 and 2
    * Note that we have only changed 3 lines of code compared to the snippet above


<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

In [ ]:
num_samples = 10000
num_islands = 10
q_init  = 1 # could be any value between (1, num_islands)
samples = [q_init] # add init value to the beginning of the trace

# Probability proportional to island ID and 0 if islands outside the range
prior = lambda x: 4 if x in range(1,3) else 1
likelihood = lambda x: x if x in range(1,num_islands+1) else 0
P = lambda x: likelihood(x)*prior(x)

for i in range(num_samples):
    # current island
    q = samples[i] 

    # Randomly propose a new state (proposal)
    proposal = 1 if np.random.uniform(0,1) > .5 else -1
    q_prop = q + proposal

    # Move to proposed state? 
    q_next = q_prop if np.random.uniform(0,1) <= P(q_prop)/P(q) else q
    samples.append(q_next)

# construct an inference data object
dataset = xr.Dataset(
    {
        "θ": (["chain", "draw"], np.array([samples]))
    },
    coords = {
        "chain": (["chain"], np.arange(1)),
        "draw": (["draw"], np.arange(num_samples+1))
    }
)
trace = az.InferenceData(posterior=dataset)
pm.plot_trace(trace);

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Beta-Bernoulli model in PyMC

* We introduce a new model (Beta-Bernoulli) that has been commonly used for estimating the probability of some event

\begin{align*}
y &\sim \mathrm{Bernoulli}(p=\theta) \\
\theta &\sim \mathrm{Beta}(\alpha=1, \beta=1)
\end{align*}

* In this model, $y$ is a binary vector with the results of $N$ coin tosses (or occurrences of an event of interest)

* We will use this model to emphasize some of the pitfalls of Metropolis

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Beta distribution

* It is a common distribution for parameters whose support is in $(0,1)$

* Given $\alpha, \beta \in \mathbb{R}$, the density function is
$$
\mathrm{beta}(\theta \mid \alpha, \beta) = \frac{\theta^{(\alpha-1)} (1-\theta)^{(b-1)} }{\int^1_0 \theta^{(\alpha-1)} (1-\theta)^{(b-1)} dx}
$$


* The distribution is handy because it is a **conjugate prior** of Bernoulli, Binominal, Negative Binomial and Geometric distributions (more about this later)

* Conjugacy with Bernoulli

    * Consider a prior $P(\theta) = \mathrm{beta}(\theta \mid \alpha,\beta)$
      
    * Consider a likelihood $ P(\{y_i\}_{i \in 1 .. N} \mid \theta) = \prod_{i \in 1 .. N } \mathrm{Bernoulli}(y_i \mid \theta)$ for $i \in 1..N$; which are results of coin tosses.
    
    * One can prove that the posterior equals $P(\theta \mid \{y_i\}_{i \in 1..N}) = \mathrm{beta}(\theta \mid \alpha=\alpha+z, \beta=N-z+\beta)$ where $z$ denotes the number of 1s in $\{y_i\}_{i \in 1..N}$
    
    * For a proof see Kruschke Ch. 7.4.2.
 
* Below we show examples of Beta distributions with different $\alpha$ and $\beta$ parameters

In [ ]:
fig, ax = plt.subplots(5,5, figsize=(10,10))
fig.tight_layout()
fig.subplots_adjust(top=0.96)

params = [0.1, 0.5, 1, 2, 3]
domain = np.linspace(0.0, 1.0, 100)
for i in range(0, 5):
    for j in range(0, 5):
        image = [ scipy.stats.beta.pdf(theta,a=params[j],b=params[i]) for theta in domain ]
        ax[i][j].plot(domain, image, linewidth=2)
        ax[i][j].set_title(f'α={params[j]} β={params[i]}')
        ax[i][j].axis([-0.02, 1.02, -0.02, 3.02])
plt.show();

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

Here we generate some data and define the model in PyMC

In [ ]:
data = xr.Dataset(
    {'tosses': ('index', [1]*14 + [0]*6)}, 
    coords={'index': np.arange(20)}
)

with pm.Model() as beta_binomial_MH:
    θ = pm.Beta(name='θ', alpha=1, beta=1) # uniform prior
    y = pm.Bernoulli(name='obs', p=θ, observed=data.tosses)

In [ ]:

pm.model_graph.model_to_graphviz(beta_binomial_MH)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Effect of Variance on Proposal Distribution

* In King Markov example, he could only move to adjacent islands
    
    * **[Question]: Would it be beneficial if he could jump to further islands directly?**


* We explore the effect proposal distributions with difference variance on the result of Metropolis


* Since $\theta$ is a continuous parameter in the Beta-Bernoulli model, we need a continuous proposal distribution


* It is standard to use a Normal distribution with $\mu=0$ with variable standard deviation $\sigma$:
$
q_{\textrm{prop}} \sim \mathcal{N}(0, \sigma)
$


* The value of $\sigma$ will determine the variance of the proposals ($\sigma^2$)
    
* Instead of implementing our own Metropolis, we use PYMC (and configure it)

* In general, a covariance matrix is given to Metropolis in PYMC, but we only have one parameter, so it is a single cell.

In [ ]:
with beta_binomial_MH:
    step_002  = pm.step_methods.Metropolis(S=np.array([0.02]))
    trace_002 = pm.sample(50_000, 
                          step=step_002, # define sampling algorithms
                          chains=1,      # number of chains
                          tune=0,        # number tunning steps (explained below)
                          initvals={'θ': 0.01} # initial state `q_init`
                         )
    
    step_02   = pm.step_methods.Metropolis(S=np.array([0.2]))
    trace_02  = pm.sample(50_000, step=step_02, chains=1, tune=0,initvals={'θ': 0.01})
    
    step_2    = pm.step_methods.Metropolis(S=np.array([2]))
    trace_2   = pm.sample(50_000, step=step_2, chains=1, tune=0,initvals={'θ': 0.01})

traces = [trace_002, trace_02, trace_2]

In [ ]:
(fig, axs) = plt.subplots(2,3, figsize=(15,6))

for (trace, i) in zip(traces, range(3)):
    for ((init,end), j) in zip([(0,100), (49900, 50000)], [1,0]):
        θ_value = trace.posterior.θ.sel(chain=0)[init:end]        
        ax = axs[j,i]
        ax.plot(θ_value, np.arange(init,end),'.-')
        ax.xaxis.set_ticks(np.linspace(0, 1, num=5))
        ax.set_title(f'σ={2/(10**(2-i))}')

axs[1,0].set_xlabel('θ')
axs[1,0].set_ylabel('time step');

* Note that, in top row, the chains show samples closer to the mode

    * This means that they are closer to the high density area
 
* In the bottom row, the chain moves towards the high density area
    
    * The samples required during this process are harmful to the chain, as they are not representative samples
    
    * They are simply the result of a bad starting state

    * Paths slowly moving towards high density areas also occur when probability density is widespread
 
* This is a common phenomenon, as the chain may start in a state far from the high density area


* This is the reason why Metropolis algorithm discard **burn-in** or **tune** samples.
    * Recall the parameter `tune` above. We should set it to larger than 0 for these types of algorithms.
    * It is common to set 500 or 1000 tuning samples.

In [ ]:
for (trace, i) in zip(traces, range(3)):
    ax = pm.plot_trace(trace)
    ax[0][0].set_title(f'σ={2/(10**(2-i))}')

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Correlated samples

* One of the issues with the Metropolis algorithm is that nearby samples are correlated

* This is a consequence of the way proposals are generated; as a shift from the current position

* As a consequence, set of nearby samples do not add much information about the overall posterior distribution

    * In other words, the accuracy of the chain is low

* **Autocorrelation** measures the correlation of chain values with the chain values $k$ steps ahead; the number of steps ahead is known as *lag*

* Below we plot the autocorrelation for the traces above for lags 0-100

* The problem for $\sigma=0.02$ above is that samples are heavily correlated
    
* There is a risk that they do not provide an accurate representation of the posterior (many samples would be needed)

In [ ]:
(fig, axs) = plt.subplots(1,3, figsize=(17,3))
for (trace, i) in zip(traces, range(3)):
    pm.plot_autocorr(trace, ax=axs[i])  
    axs[i].set_title(f'σ={2/(10**(2-i))}')

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Effective Sample Size (ESS)

* **Effective Sample Size** (ESS) summarizes autocorrelation information within a chain

* ESS quantifies how much non-autocorrelated information a chain contains
    
    * That is, the amount of samples that provide additional information about the posterior

* Let $N$ denote the length of the chain and $\mathrm{ACK}(k)$ the autocorrelation of the chain at lag $k$. The ESS is defined as:

$$
\mathrm{ESS} = \frac{N}{1+2\sum^\infty_{k=1}\mathrm{ACK}(k)}
$$


* **Good ESS values should approach the number of generated samples**

    * This means that each sample in the chain contributes to better understand the posterior distribution



* Below we compute ESS for the traces above

    * As mentioned earlier, all the chains above show a low ESS; especially $\sigma=0.02$

In [ ]:
print(f'ESS (σ=0.02): {pm.ess(trace_002).θ.item()} out of {trace_002.posterior.θ.size}')
print(f'ESS (σ=0.2):  {pm.ess(trace_02).θ.item()} out of {trace_02.posterior.θ.size}')
print(f'ESS (σ=2):    {pm.ess(trace_2).θ.item()} out of {trace_2.posterior.θ.size}')

* PyMC shows this warning sometimes: *The number of effective samples is smaller than 25% for some parameters.*
    * **[Question]: What does this mean?**
        

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

#  Gibbs Algorithm

* A modification of Metropolis
    
    * Uses a proposal distribution whose samples are always accepted
        
    * Exploits the marginals of the posterior distribution as proposals
    
    * It applies to posterior distributions with more than 1 dimension

* *Only applicable when we have the analytical form of the marginals of the posterior* (so conjugate priors for marginals)

* To sample from the posterior $P(\theta_1, \theta_2, \ldots, \theta_n \mid D)$, we must be able to sample from $P(\theta_1 \mid \{\theta_i\}_{i \not= 1}, D), P(\theta_2 \mid \{\theta_i\}_{i \not= 2}, D), \ldots, P(\theta_n \mid \{\theta_i\}_{i \not= n}, D)$

* Requires that the model is built using **conjugate priors**
    
    * "If the posterior distribution $P(\theta \mid D)$ is in the same probability distribution family as the prior probability distribution $P(\theta)$, the prior and posterior are then called *conjugate distributions*, and the prior is called a *conjugate prior* for the likelihood function $P(D \mid \theta)$." [Source: Wikipedia]

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

### 2D Beta-Bernoulli

* Consider the Beta-Bernoulli example above, but for infering the posterior bias of two independent coins:

$$
\begin{align*}
y_2 &\sim \mathrm{Bernoulli}(p=\theta_2) \\
y_1 &\sim \mathrm{Bernoulli}(p=\theta_1) \\
\theta_2 &\sim \mathrm{Beta}(\alpha=1, \beta=1) \\
\theta_1 &\sim \mathrm{Beta}(\alpha=1, \beta=1)
\end{align*}
$$

* Let $y_i$ be binary vectors with the result of 20 coin tosses for the two independent coins
    
    * Furthermore we use $z_i$ to denote the number of 1s in $y_i$


* The posterior marginals for this model are (where $N_1 = N_2 = 20$)

    * $P(\theta_1 \mid \theta_2, \{y_1, y_2\}) = \mathrm{beta}(\theta_1 \mid z_1 + \alpha_1, N_1 − z_1 + \beta_1)$
    
    * $P(\theta_2 \mid \theta_1, \{y_1, y_2\}) = \mathrm{beta}(\theta_2 \mid z_2 + \alpha_2, N_2 − z_2 + \beta_2)$
 
    * For a proof, see Kurschke Ch. 7.4.4.
 

* Below we show an implementation of Gibbs for the problem
    * Note the similarities with the Metropolis implementations above

In [ ]:
# generate date for example
data = xr.Dataset(
    {
        'tosses1': ('index', [1]*14 + [0]*6), 
        'tosses2': ('index', [1]*19 + [0]*1)
    },
    coords={'index': np.arange(20)}
)

# Gibbs implementation starts here
num_samples = 1000
samples = []
q_init = (0.01, 0.01)
samples.append(q_init)

# target distribution
# not needed here, it is embedded in the proposal (see below)

for i in range(num_samples):
    # Randomly propose a new state (proposal)
    # Important: we can calculate the priors of the marginal using the beta conjugacy
    # We still do not know the shape of the bivariate posterior (which is approximated with Gibbs here)
    θ1_prop = np.random.beta(a=sum(data.tosses1)+1, b=data.tosses1.size - sum(data.tosses1) + 1)
    θ2_prop = np.random.beta(a=sum(data.tosses2)+1, b=data.tosses2.size - sum(data.tosses2) + 1)

    # We move to new state with probability 1!
    samples.append((θ1_prop, θ2_prop))

# Gibbs implementation finishes here

# construct an inference data object
dataset = xr.Dataset(
    {
        "θ1": (["chain", "draw"], np.array([[s[0] for s in samples]])),
        "θ2": (["chain", "draw"], np.array([[s[1] for s in samples]]))
    },
    coords = {
        "chain": (["chain"], np.arange(1)),
        "draw": (["draw"], np.arange(num_samples+1))
    }
)
trace_GB = az.InferenceData(posterior=dataset)

In [ ]:
pm.plot_trace(trace_GB);

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

* To better illustrate the performance improvement of Gibbs, we compare it with sampling using metropolis

* The code below creates the mode in PyMC and samples with Metropolis
    * We generate the same number of samples and include the tunning samples for the purpose of the comparison
    * We set the same initial state in both algorithms

In [ ]:
with pm.Model() as beta_bernoulli:
    θ1 = pm.Beta(name='θ1', alpha=1, beta=1)
    θ2 = pm.Beta(name='θ2', alpha=1, beta=1)
    y1 = pm.Bernoulli(name='obs1', p=θ1, observed=data.tosses1)
    y2 = pm.Bernoulli(name='obs2', p=θ2, observed=data.tosses2)

with beta_bernoulli:
    step_MH  = pm.step_methods.Metropolis()
    trace_MH = pm.sample(1000, step=step_MH, chains=1, tune=0, initvals={'θ1': 0.01, 'θ2': 0.01})

* Below we show a pair plot of the samples

* Note how Metropolis needs to "walk towards" the high density area in the first samples

In [ ]:
(fig, axs) = plt.subplots(1,2, figsize=(12,5))
pm.plot_pair(trace_MH, ax=axs[0]);
axs[0].set_title('Metropolis');
pm.plot_pair(trace_GB, ax=axs[1]);
axs[1].set_title('Gibbs');

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

* Now we compute the ESS for both chains

* We observe a clear superiority of Gibbs
    * Gibbs achieves an ESS almost equal to the total number of samples
    * Metropolis archives around 20% ESS

In [ ]:
print(f'\nMetropolis \n {pm.ess(trace_MH)}')
print(f'\nGibbs \n {pm.ess(trace_GB)}')

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Metropolis and Gibbs limitations

* Both Metropolis and Gibbs suffer in high-dimensional spaces

* The reason behind this phenomenon is called *concentration of measure*

* Metropolis and Gibbs are good at exploring distributions where all probability density is allocated in the area around the mode

* However, the higher the dimensionality of a model, the further from the mode are the largest portions of probability density

<img style="float: right; height: 300px" src="multivariate-normal-example.png" alt="Source Wikipedia: https://en.wikipedia.org/wiki/Multivariate_normal_distribution#/media/File:MultivariateNormal.png">

* The figure below illustrates this phenomenon using a multivariate Gaussian distribution with increasing dimensionality
    
    * We start with a univariate Gaussian $X \sim \mathcal{N}(0,1)$
    
    * Then a 10-dimension multivariate Gaussian $X_1, X_2, \ldots, X_{10} \sim \mathcal{N}(\vec{0},\Sigma)$ where $\vec{0}$ is a ten dimensional zero vector, and $\Sigma$ is a 10x10 diagonal matrix with all 1s in the diagonal.

    * We also consider 100 and 1000 dimensions
    
    * The plot shows a radial distance of a random sample to the mode (which is the origin for all examples)
 
* It is easy to see that the larger the dimensionality the larger the distance
    * As a result, it is likely that Metropolis and Gibbs will perform poorly in high-dimensional models



In [ ]:
Ds = [1,10,100,1000]
colors = ['red','blue','green','brown','purple']
T = int(1e3)
rad_dist = lambda Y : np.sqrt(np.sum(np.power(Y,2)))


for (D,c) in zip(Ds, colors):
    np.zeros(D)
    Y = np.random.multivariate_normal(mean=np.zeros(D), cov=np.diag(np.ones(D)), size=T)
    rad_distances = np.array([rad_dist(y) for y in Y])
    az.plot_dist(rad_distances, label=f'{D}', color=c);

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Hamiltonian Monte-Carlo

<img style="float: right; height: 250px; padding-left: 1cm;" src="william-rowan-hamilton.jpeg" alt="Source Wikipedia: https://en.wikipedia.org/wiki/Hamiltonian_mechanics#/media/File:WilliamRowanHamilton.jpeg">

* HMC is more computationally costly than Metropolis or Gibbs, but its **proposals are more efficient.**  
* HMC does not need as many samples to describe the posterior. 
* This may save total computing time.

* Hamiltonian MC assumes we can **approximate the gradient of the posterior** around the current sampling point
* **Exploits the local shape of the posterior** curve/surface/etc.
* Uses physics simulation to follow steep gradients faster.

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# HMC: Intuition

<img src="hmc.png" width="52%" style="margin-left: auto; margin-right: auto">

* Top row: we do not want to sample symmetrically from the two starting points. We want to skew for the distribution.
* Let's invert the density function (negative logarithm, known as **potential**)
* Intuition: potential is like **gravitational potential** in physics
    * potential energy can be translated to **kinetic energy**
* Imagine that current position is a particle (a point mass) 
    * In a limited time it will roll down the potential curve/surface, etc.
* To introduce randomness we apply a **random momentum** and let it roll, and see where it lands.
    * random = gaussian centered on current position
    * This allows the gradient to **weight** the sampling.
    * The lower we go, the higher posterior density, so we get what we need.
* To compute the physical transfer of gravitational energy to speed, we need to know the gradient (the **differential**) of the posterior.
  * This is why everything is based on the Pytensor library which represents computations that are **differentiable** (or TensorFlow)
  * The book does not show the physical calcuation, see supplementary material [MacKay, Section 30.1, p. 387; pdf-page 399](https://inst.eecs.berkeley.edu/~ee121/sp08/handouts/it.pdf)
* Bottom row: the sampled 'proposal distributions'
    * The proposal is not symmetric
    * It works in multiple dimensions
    * The cost of each sampling is much higher, but the autocorrelation decreases

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# HMC: Another Figure  (McElreath Ch. 9)

<img src="hmc-figure9.6.png" style="width: 55%;">
(McElreath, Ch. 9)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# HMC: Simulation Visualization

* Visit this website to visualize the steps of different MCMC algorithms
    * Hamiltonian MC: http://chi-feng.github.io/mcmc-demo/app.html?algorithm=HamiltonianMC&target=banana
 
<img style="width: 500px" src="skatepark.jpg">

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# A Hamiltonian

* Hamilton has shown that we can describe any mechanical system with just 2 variables (vectors): position $\mathbf x$ and momentum $\mathbf y$.

* Hamiltonian is a constant vector expressing the total energy in the system. Let $\mathbf{x}$ denote position, and let $\mathbf{p}$ denote momentum (both are vectors as we are in a multi-dimensional space).  Then the following sum is constant:
$$H(\mathbf x,\mathbf p) = E (\mathbf x) + K (\mathbf p)$$
  where $E$ stands for potential energy, and $K$ stands for kinetic energy.  For instance, $K(\mathbf p) = \mathbf p^\mathsf{T}\mathbf p/2$.  
  
* In the symbols used in our context:
$$H(\mathbf \theta,\mathbf \phi) = E (\mathbf \theta) + K (\mathbf \phi)$$

* While keeping the energy constant we explore slower (more precisely) the areas with low potential values and faster (less precisely) the areas with high potential values, but gravity has a tendency to lead us to low potential energy (high density).  

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Hamiltonian MC, Formal Definition

* Assume that we can write the probability distribution in the following format, where $Z$ is a constant:

$$ P(\mathbf \theta) = \frac{\mathrm e^{-E (\mathbf \theta)}}{Z} $$

      
* Then the lowest values are the __highest density__ values

* Then we can not only calculate the probability density $P$ but also its differential (as $\mathrm{e}^{x}\mathrm dx = e^x$), if we can calculate $E(x)$ and its differential

* This is one reason why PyMC works with log-probabilities: $P(x) = \mathrm e^{\mathrm{logp}(x)}$, and why Pytensor is used to represent functions (these functions are $E$ here)

* For each parameter $\theta_j$ in the parameter space, Hamiltonian MC adds a momentum variable $\phi_j$ and we sample from the joint posterior:

  - For every point in the posterior, we consider a valuation of the position $\theta$ (capturing the potential energy, or here the probability density) and 
  - A valuation for the potential $\phi$ (capturing the kinetic energy, so how fast are we moving through the space).
  

* In a multi-dimensional space both are vectors

* Like in physics the size of jump (a proposal) for $\theta$ is decided by the momentum $\phi$. Bigger the momentum, further we will move.

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

## The Algorithm

* We alternate the moves: 
  - Given the current point $\theta$ decide a new proposal for momentum - this is done by choosing a random momentum with a proposal distribution for a given position $\mathbf\theta$, for instance Gaussian
  - Given a momentum compute a proposal for $\theta$ - this is done using deterministic simulation, and uses a differential of $K(\phi)$
  - Since both distributions for the parameters and for the potentials are multivariate we are moving in a multidimensional space, reflecting the gradients in all directions
  - Since the energy has to be preserved, if $H(\theta, \phi)$ changes, it means that we have found a numerical calculation error, and no longer follow the shape of our function (__reject the proposal__)


* We obtain a simulation for both vectors of variables, but then we can simply ignore the potentials, to get the estimation for the parameters in $\mathbf\theta$.

* In the algorithm below (from MacKay) **x** is a parameter vector, and **p** is the potential vector.

* The algorithm below uses the Leapfrog method to run the physics simulation
    * Apparently making two half-steps increases precision (see algo below), a result from automatic integration theory (only for the interested: https://bayesianbrad.github.io/posts/2019_hmc.html) .
  
<img src="./algorithm.png" style="width: 90%">

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

## HMC a few details

* HMC works for continuous variable domains
* We would accept all proposals if calculations were exact (because the sum of kinetic and potential energy are constant)
* Approximations of Hamiltonian simulation mean that sometimes we need to reject, as we do not want imprecisions to affect the samples.

* The acceptance condition is like in the metropolis (ratio of posteriors, but corrected by the prior probability of the random momentum $\phi$

$$
p_\mathrm{accept} = \min \left({{p(\theta_\mathrm{proposed} | D) p(\phi_\mathrm{proposed})}\over{p(\theta_\mathrm{current} | D) p(\phi_\mathrm{current})}}, 1\right)
$$

* Note that the in our simulation the sum of the logs of the terms in the numerator/denomminator is the total energy system (posterior is the potential energy, and the prior on the momentum is the kinetic energy).  
    * In physics this sum is constant if there is no energy loss
    * But the sum of logarithms equals to the logarithm of a multiplication, so the logarithm of the multiplication in the numerator/denominator is a constant.
    * But if a logarithm is constant  then so is its argument.
    * So the ratio should always be equal to 1 if we had perfect maths and perfect simulation
    * In practice this rules catches imprecisions of calculations.

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

## HMC: Parameters

<img style="float: right;padding-left: 4cm;" width="60%" src="overshooting.png">


* **Epsilon** ($\epsilon$)
  * The size of the simulation step 
  * Smaller steps, in principle, mean better precision, and less rejections
  * During a step the particle moves linearly. If the step size is small, then the particle can turn sharply. If the step size is large, then each leap will be large and could even overshoot the point where the simulation would want to turn around.
* **Steps** ($\tau$)
  * The number of simulation steps. 
  * These control time. 
  * Too few steps, we move too slowly; 
  * Too many steps we may overshoot the high density locations
* **Standard deviation** of the initial momentum
* **Warm up period**
    *  Set an acceptance rate. The book recommends 65%. PyMC API recommends 95%.
    *  Modern HMC implementations will run a warm-up stage to automatically determine: i) standard deviation of the initial momentum, ii) $\epsilon$ and iii) $\tau$ so that generated samples match the acceptance rate.
    * PyMC method `sample` takes an argument ``target_accept: float in [0, 1]``.
        * The step size is tuned automatically to get this acceptance rate.
        * The `tune` parameter for HMC is actually tuning, it is not producing (in principle) useful samples that are thrown away.  It tries to understand the scale and gradient to choose a good step size.  This warm-up tends to be slower than in other sampling methods.

* Figure: examples of overshooting with long trajectories (steps $\times$ epsilon)
    * Note how bad the posterior approximation is in the right column,  bottom plot
  
* Note that the long trajectories make a U-turn
    * **NUTS** = no U-turn sampler (Hoffman & Gelman, 2014) 
    * NUTS has a protection against these U-turns to improve effectiveness
    * This is the Hamiltonian sampler we used most of the time, as it is default in PyMC

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# HMC complains when things go wrong

* Unlike in Metropolis or Gibbs, it is possible to detect errors during Hamiltonian MC sampling

* The nature of the algorithm allows us to determine when the simulation is going wrong

* Here are some example warning reported by PyMC
    
    * **Target accept not reached**. If the simulation cannot reach the target accept during warm-up, then it will be reported in a warning.
    
    * **Divergences**. HMC can detect when the simulation goes off rails (too far from the actual gradient of the negative log-density). It checks whether the value of the Hamiltonian is too far from the previous state. Those simulations are called *divergent*. PyMC reports the number of divergent transitions during sampling.
        
    * Divergences sometimes originate due to the curvature of the posterior. That is, they are more likely if the posterior has sharp curves
          
 
* **FOLK THEOREM OF STATISTICAL COMPUTING (by Andrew Gelman)**: *When you have computational problems, often there is a problem with your model.*

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# Diagnostics

* Although all the above algorithms converge to a unique stationary distribution in the limit, we can sample forever. Therefore, we need tools to determine whether a sample has some problems
    * Please note that the metrics and guidelines below aim at finding problems with the generated trace, not at ensuring that the trace has converged and it is an accurate description of the posterior


* **Diagnostics have two main goals:**    
    * Determine that sampling has *converged*      
        * This is done by generating and comparing multiple chains    
    * Determine that the *accuracy* of the sample and whether it is *representative* of the posterior
        * This is done by checking the correlation between samples



* Below we sample from artificially odd models. We will use these traces to illustrate issues in sampling.


In [ ]:
with pm.Model() as hmc_divergent:
    σ = pm.Exponential('σ', 0.001)
    α = pm.Normal('α',0,1000)
    y = pm.Normal('y',mu=α, sigma=σ, observed=[-1,1])
    trace_hmc_divergent = pm.sample(1000, chains=4)
    step = pm.step_methods.Metropolis()
    trace_mh_divergent = pm.sample(1000,step=step,chains=4,tune=0) # artificially bad

with pm.Model() as hmc_fixed:
    σ = pm.Exponential('σ', 1)
    α = pm.Normal('α',0,10)
    y = pm.Normal('y',mu=α, sigma=σ, observed=[-1,1])
    trace_fixed = pm.sample(1000, chains=4, target_accept=0.95)

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

## Visualization

* Visual checks are useful to determine problem in convergence and representativeness

* To this end, a *trace plot* is a very useful tool

* To check for convergence, we should observe that all chains in the trace plot overlap
    * In the example, below we observe at least one chain which is not overlapping with the rest

* The representativeness of the distribution is also low
    * The right-hand-side, shows little mixing of the values in the chain

In [ ]:
pm.plot_trace(trace_mh_divergent, var_names=['α']);

* On the contrary, the trace below shows great overlap of all chains and  very good mixing or the right-hand-side

In [ ]:
pm.plot_trace(trace_fixed, var_names=['α']);

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# ESS & Monte Carlo Standard Error (MCSE)

* We have already discussed the Estimated Sample Size (ESS) as a metric of the number of effective (non-correlated) samples within a chain

* This is a good metric of the representativeness and accuracy of a chain
    * Depending on the goal of the analysis－e.g., estimating the mode or estimating the HDI－the textbook suggests that we might need around 500 effective samples

* Since ESS is estimated using a sample, it is a noisy estimate.

* To understand how noisy our estimates using the sample chain are we can use the *Monte Carlo Standard Error* (MCSE)

* Let $\mathrm{SD}$ denote the standard deviation in the chain, MCSE is defined as

$$
\mathrm{MCSE} = \frac{\mathrm{SD}}{\sqrt{\mathrm{ESS}}}
$$

* A good chain should show values of MCSE very close to 0.0. In such a case, estimates of the mode of the distribution may be accurate even with low ESS.

* Again, MCSE is a measure of the accuracy in the chain

In [ ]:
pm.summary(trace_mh_divergent)

In [ ]:
pm.summary(trace_fixed)

## $\hat{R}$

* $\hat{R}$ is a measure of convergence
    * Details at  [Vehtari et al. "Rank-normalization, folding, and localization:
An improved $\hat{R}$ for assessing convergence of MCMC"](https://arxiv.org/abs/1903.08008)

* As a consequence, it requires multiple chains to be computed

* Let $M$ denote the number of chains, $N$ number of samples per chain, $\theta^{(nm)}$ sample $n$ in chain $m$ for parameter $\theta$,. The split-$\hat{R}$ (simplified version of what PyMC computes) is defined as:

* Between chains variance
$$
B = \frac{N}{M-1}\sum^M_{m=1}(\bar{\theta}^{(.m)} - \bar{\theta}^{(..)})^2 ~ \text{ where } ~ \bar{\theta}^{(.m)} = \frac{1}{N} \sum^N_{n=1}\theta^{(nm)} ~ \text{ and } ~ \bar{\theta}^{(..)} = \frac{1}{M} \sum^M_{m=1}\theta^{(.m)}
$$

* Within chain variance
$$
W = \frac{1}{M}\sum^M_{m=1}s^2_m ~ \text{ where } ~ s^2_m = \frac{1}{N-1}\sum^N_{n=1}(\theta^{(nm)} - \bar{\theta}^{(.m)})^2
$$

* Marginal posterior variance
$$
\hat{\mathrm{var}}^{+}(\theta \mid y) = \frac{N-1}{N}W + \frac{1}{N}B
$$

* Finally,
$$
\hat{R} = \sqrt{\frac{\hat{\mathrm{var}}^+(\theta \mid y)}{W}}
$$


* Importantly, $\hat{R}$ should approach one from above. That is, converging chains will have a $\hat{R}$ value close to 1

* Below we plot the $\hat{R}$ for the poorly converging trace and the good converging one. We clearly observe values far from 1.0 for the former and values close to 1.0 for the latter.

In [ ]:
pm.rhat(trace_mh_divergent).items()

In [ ]:
pm.rhat(trace_fixed).items()

<br/><br/><br/><br/><br/><br/><br/><br/><br/><br/>

# MSc Theses on Markov Chain Monte Carlo

* Did you like this topic? Would you like to work on **MSc thesis project on MCMC algorithms**?

    * **Contact us! Andrzej (wasowski@itu.dk) or Raúl (raup@itu.dk)**


* We working on improving the efficiency and diagnostics for MCMC algorithms, for example on discrete problems with hard constraints with help of SAT/SMT-solving